# QUERY EXPANSION

# Import Required Libraries

Initialize all dependencies required for vector search, embeddings, LLM interaction, and environment configuration.

In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime

python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line

Load environment variables and initialize the Groq client for LLM inference.

In [2]:
from dotenv import load_dotenv
load_dotenv()

python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7


True

In [3]:
from groq import Groq

In [4]:
import os
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [5]:
client=Groq()

In [6]:
client

In [7]:
model_name = 'openai/gpt-oss-120b'

# Initialize Embedding Model

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Mohd Zaid\AppData\Local\Temp\ipykernel_4348\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [9]:
pdf_folder_location = "tesla-annual-reports"

In [10]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [11]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [12]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [ ]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(tesla_10k_chunks)

3337

In [ ]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
chromadb_client.heartbeat()

1780556845094276000

In [ ]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


# Chunking

In [ ]:
i = 0

while i < len(tesla_10k_chunks):
    vectorstore.add_documents( 
        documents=tesla_10k_chunks[i:i+500], 
        ids=["text_" + str(i) for i in range(i, i+500)] 
    )

    i += 500 
    time.sleep(5)

In [ ]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
collection = chromadb_client.get_collection(tesla_10k_collection)

In [ ]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023']

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        'k': 5
    }
)

# Query Expansion Strategy

In [ ]:
query_expansion_system_message = """
You are a financial analyst assistant specialized in SEC 10-K filings.

Generate exact 4 query expansions for the user query.

The expanded queries MUST preserve the original intent.

Generate query variants from these perspectives:

1. Financial analyst phrasing
2. Risk-factor phrasing
3. Operational/business phrasing
4. Synonym/subtopic phrasing
5. Strategic phrasing
6. Financial performance phrasing

Rules:

* Do NOT generate unrelated questions
* Keep the meaning grounded to the original query
* Make each query semantically different
* Queries should help retrieve different relevant chunks from a 10-K filing

Return ONLY the expanded queries.
One query per line.
No numbering.
No bullets.
No explanations.
"""

user_message_template = """ <Question>
{question} </Question>
"""


In [ ]:
user_input = "Any specific fines levied on the company in 2022?"

In [ ]:
benchmark_questions = {
"Q1": "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A.",

"Q2": "Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures. Avoid generic AI claims.",

"Q3": "A supplier asks whether Tesla is exposed to concentration risk across factories, suppliers, raw materials, or geographies. Prepare a concise risk note with citations.",

"Q4": "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?"

}
question_id = "Q1"

user_input = benchmark_questions[question_id]

In [ ]:
baseline_docs = retriever.invoke(user_input)

baseline_top_chunks = []

for rank, doc in enumerate(baseline_docs):

    baseline_top_chunks.append({
        "chunk_id": f"baseline_{rank}",
        "content": doc.page_content,
        "section": doc.metadata.get("source", "unknown"),
        "score": 1 / (rank + 1)
    })

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
prompt=[
    {'role':'system','content':query_expansion_system_message},
    {'role':'user','content':user_message_template.format(
        question=user_input
    )}
]

In [ ]:
query_expansions=client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=prompt,
    temperature=0
)

In [ ]:
print(query_expansions.choices[0].message.content)

What are the key external supply chain risks that could impact Tesla's growth prospects as disclosed in the Risk Factors section of the 10-K filing?

How does Tesla's internal cost structure and execution compare to its peers in terms of scalability and efficiency, as discussed in the MD&A section of the 10-K filing?

What are the primary operational risks that could hinder Tesla's ability to meet growing demand, as outlined in the Risk Factors section of the 10-K filing?

How does Tesla's management discuss the trade-off between investing in growth and maintaining profitability, and what are the implications for the company's cost structure and execution, as presented in the MD&A section of the 10-K filing?

What are the key performance indicators (KPIs) used by Tesla to measure its internal execution and cost structure, and how do these metrics compare to industry benchmarks, as discussed in the MD&A section of the 10-K filing?

What are the external factors that could impact Tesla's

In [ ]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [ ]:
query_expansions_list

["What are the key external supply chain risks that could impact Tesla's growth prospects as disclosed in the Risk Factors section of the 10-K filing?",
 '',
 "How does Tesla's internal cost structure and execution compare to its peers in terms of scalability and efficiency, as discussed in the MD&A section of the 10-K filing?",
 '',
 "What are the primary operational risks that could hinder Tesla's ability to meet growing demand, as outlined in the Risk Factors section of the 10-K filing?",
 '',
 "How does Tesla's management discuss the trade-off between investing in growth and maintaining profitability, and what are the implications for the company's cost structure and execution, as presented in the MD&A section of the 10-K filing?",
 '',
 'What are the key performance indicators (KPIs) used by Tesla to measure its internal execution and cost structure, and how do these metrics compare to industry benchmarks, as discussed in the MD&A section of the 10-K filing?',
 '',
 "What are the 

In [ ]:
from collections import defaultdict

fusion_scores = defaultdict(float)

chunk_metadata = {}

for query in query_expansions_list:

    docs = retriever.invoke(query)

for rank, doc in enumerate(docs):

    chunk_text = doc.page_content

    # Reciprocal Rank Fusion
    fusion_scores[chunk_text] += 1 / (rank + 1)

    if chunk_text not in chunk_metadata:

        chunk_metadata[chunk_text] = {
            "retrieved_by": [],
            "section": doc.metadata.get("source", "unknown")
        }

    chunk_metadata[chunk_text]["retrieved_by"].append(query)


In [ ]:
reranked_chunks = sorted(
fusion_scores.items(),
key=lambda x: x[1],
reverse=True
)

expanded_top_chunks = []

for idx, (chunk, score) in enumerate(reranked_chunks[:5]):


    expanded_top_chunks.append({
        "chunk_id": f"expanded_{idx}",
        "content": chunk,
        "section": chunk_metadata[chunk]["section"],
        "score": score,
        "retrieved_by": chunk_metadata[chunk]["retrieved_by"]
})


final_context_documents = "\n\n".join(
[
f"[{x['chunk_id']}]\n{x['content']}"
for x in expanded_top_chunks
]
)

In [ ]:
len(final_context_documents)

7457

In [ ]:
final_context_documents

'[expanded_0]\nclaims\tfor\tvehicle\texposure,\tmeaning\tthat\tany\tproduct\tliability\tclaims\twill\tlikely\thave\tto\tbe\tpaid\tfrom\tcompany\tfunds\tand\tnot\tby\tinsurance.\nWe\twill\tneed\tto\tmaintain\tpublic\tcredibility\tand\tconfidence\tin\tour\tlong-term\tbusiness\tprospects\tin\torder\tto\tsucceed.\nIn\torder\tto\tmaintain\tand\tgrow\tour\tbusiness,\twe\tmust\tmaintain\tcredibility\tand\tconfidence\tamong\tcustomers,\tsuppliers,\tanalysts,\tinvestors,\tratings\nagencies\tand\tother\tparties\tin\tour\tlong-term\tfinancial\tviability\tand\tbusiness\tprospects.\tMaintaining\tsuch\tconfidence\tmay\tbe\tchallenging\tdue\tto\tour\tlimited\noperating\thistory\trelative\tto\testablished\tcompetitors;\tcustomer\tunfamiliarity\twith\tour\tproducts;\tany\tdelays\twe\tmay\texperience\tin\tscaling\tmanufacturing,\ndelivery\tand\tservice\toperations\tto\tmeet\tdemand;\tcompetition\tand\tuncertainty\tregarding\tthe\tfuture\tof\telectric\tvehicles\tor\tour\tother\tproducts\tand\tservices;\t

In [ ]:
system_message_query_expansion="""
You are an assistant to a financial services firm.

Answer ONLY using the provided context.

Requirements:

* Use evidence from the documents
* Give analytical answers
* Include citations using [chunk_id]
* Mention operational and financial risks where relevant

If answer is not found, say "I don't know".
"""


In [ ]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [ ]:
final_prompt=[
    {
        'role':'system', 'content':system_message_query_expansion
    },
    {
        'role':'user', 'content':qna_user_message_template.format(
            context=final_context_documents,
            question=user_input
        )
    }
]

In [ ]:
response=client.chat.completions.create(
    model=model_name,
    messages=final_prompt,
    temperature=0
)

In [ ]:
print(response.choices[0].message.content)

Tesla’s growth narrative is portrayed as being far more limited by **internal execution and cost‑structure issues** than by pure external supply‑chain constraints.

**Internal execution risks**

* Tesla repeatedly stresses that its ability to scale manufacturing, delivery and service operations is a key uncertainty.  The company notes that “any delays we may experience in scaling manufacturing, delivery and service operations to meet demand” could hurt its quarterly production and sales performance and erode confidence among investors, analysts and suppliers【expanded_0】.  
* Its limited operating history and the unfamiliarity of customers with its products are highlighted as factors that make it harder to maintain credibility and to raise additional capital【expanded_2】.  
* Product‑recall risk, warranty‑reserve adequacy and the associated “significant expense, supply‑chain complications and service burdens” are described as internal cost pressures that could damage the brand and financ

In [ ]:
# final_output = {
# "question_id": question_id,


# "original_query": user_input,

# "expanded_queries": query_expansions_list,

# "baseline_top_chunks": [
#     {
#         "chunk_id": x["chunk_id"],
#         "section": x["section"],
#         "year": 2025,
#         "score": x["score"]
#     }
#     for x in baseline_top_chunks
# ],

# "expanded_top_chunks": [
#     {
#         "chunk_id": x["chunk_id"],
#         "section": x["section"],
#         "year": 2025,
#         "score": x["score"],
#         "retrieved_by": x["retrieved_by"]
#     }
#     for x in expanded_top_chunks
# ],

# "final_answer": response.choices[0].message.content,

# "citations": [
#     {
#         "chunk_id": x["chunk_id"],
#         "source_doc": x["section"],
#         "section": x["section"],
#         "year": 2025
#     }
#     for x in expanded_top_chunks
# ],

# "retrieval_improvement_analysis":
# "Expanded retrieval improved semantic coverage across financial, operational, and risk-focused terminology."

# }

# print(final_output)

{'question_id': 'Q4', 'original_query': 'Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?', 'expanded_queries': ["What are the key external supply chain risks that could impact Tesla's growth prospects as disclosed in the Risk Factors section of the 10-K filing?", '', "How does Tesla's internal cost structure and execution compare to its peers in terms of scalability and efficiency, as discussed in the MD&A section of the 10-K filing?", '', "What are the primary operational risks that could hinder Tesla's ability to meet growing demand, as outlined in the Risk Factors section of the 10-K filing?", '', "How does Tesla's management discuss the trade-off between investing in growth and maintaining profitability, and what are the implications for the company's cost structure and execution, as presented in the MD&A section of the 10-K filing?", '', 'What are

In [ ]:
all_results = []

for question_id, user_input in benchmark_questions.items():

    # baseline retrieval
    # query expansion
    # fusion
    # answer generation

    final_output = {
        "question_id": question_id,
        "original_query": user_input,
        "expanded_queries": query_expansions_list,
        "baseline_top_chunks": baseline_top_chunks,
        "expanded_top_chunks": expanded_top_chunks,
        "final_answer": response.choices[0].message.content,
        "citations": [
            {
                "chunk_id": x["chunk_id"],
                "source_doc": x["section"],
                "section": x["section"],
                "year": 2025
            }
            for x in expanded_top_chunks
        ],
        "retrieval_improvement_analysis":
        "Expanded retrieval improved semantic coverage across financial, operational, and risk-focused terminology."
    }

    all_results.append(final_output)

In [ ]:
import json

with open("assignment1_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)

print("Saved assignment1_results.json")

Saved assignment1_results.json
